# Lab Assignment 2
Data cleaning and enrichment of the Heart Failure Clinical Records dataset

## Setup

In [ ]:
!pip -q install ucimlrepo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import MinMaxScaler
from IPython.display import display

## Dataset Acquisition and Loading

In [ ]:
dataset = fetch_ucirepo(id=519)
features = dataset.data.features.copy()
target = dataset.data.targets.copy()
original_df = pd.concat([features, target], axis=1)
original_df.head()

## Dataset Summary

In [ ]:
print("Rows:", original_df.shape[0])
print("Columns:", original_df.shape[1])
display(original_df.dtypes.to_frame("Data Type"))
display(original_df.isna().sum().to_frame("Missing Values"))

## Working Dataset

In [ ]:
df = original_df.copy()
df["sex"] = df["sex"].map({0: "Female", 1: "Male"})
df["smoking"] = df["smoking"].map({0: "No", 1: "Yes"})

rng = np.random.default_rng(42)

for column in ["age", "serum_creatinine"]:
    index = rng.choice(df.index, size=6, replace=False)
    df.loc[index, column] = np.nan

index = rng.choice(df.index, size=6, replace=False)
df.loc[index, "smoking"] = np.nan

df.isna().sum()

## Visualize Missing Values

In [ ]:
plt.figure(figsize=(10, 5))
sns.heatmap(df.isna(), cbar=False, yticklabels=False)
plt.title("Missing Values")
plt.show()

## Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["age"], kde=True, ax=axes[0])
axes[0].set_title("Age")
sns.boxplot(y=df["ejection_fraction"], ax=axes[1])
axes[1].set_title("Ejection Fraction")
plt.tight_layout()
plt.show()

## Summary Before Cleaning

In [ ]:
summary_columns = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium"
]
before_stats = df[summary_columns].agg(["mean", "median", "std"]).T
before_stats

## Impute Missing Values

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

for column in ["sex", "smoking"]:
    df[column] = df[column].fillna(df[column].mode()[0])

df.isna().sum()

## Remove Outliers

In [ ]:
def remove_outliers(data, columns):
    cleaned = data.copy()
    for column in columns:
        q1 = cleaned[column].quantile(0.25)
        q3 = cleaned[column].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        cleaned = cleaned[(cleaned[column] >= lower) & (cleaned[column] <= upper)]
    return cleaned.reset_index(drop=True)

outlier_columns = [
    "creatinine_phosphokinase",
    "platelets",
    "serum_creatinine"
]

rows_before = len(df)
df = remove_outliers(df, outlier_columns)
rows_after = len(df)

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Removed:", rows_before - rows_after)

## Correct Inconsistencies

In [ ]:
invalid_age = df["age"] <= 0
invalid_ejection = (df["ejection_fraction"] <= 0) | (df["ejection_fraction"] > 100)

print("Invalid age values:", invalid_age.sum())
print("Invalid ejection fraction values:", invalid_ejection.sum())

df.loc[invalid_age, "age"] = np.nan
df.loc[invalid_ejection, "ejection_fraction"] = np.nan

df["age"] = df["age"].fillna(df["age"].median())
df["ejection_fraction"] = df["ejection_fraction"].fillna(df["ejection_fraction"].median())

## Feature Engineering

In [ ]:
def get_age_group(age):
    if age < 40:
        return "<40"
    if age <= 60:
        return "40-60"
    return ">60"

df["age_group"] = df["age"].apply(get_age_group)

ef_min = df["ejection_fraction"].min()
ef_max = df["ejection_fraction"].max()
cr_min = df["serum_creatinine"].min()
cr_max = df["serum_creatinine"].max()

ef_normalized = (df["ejection_fraction"] - ef_min) / (ef_max - ef_min)
cr_normalized = (df["serum_creatinine"] - cr_min) / (cr_max - cr_min)

df["risk_score"] = ef_normalized * cr_normalized

df[["age", "age_group", "ejection_fraction", "serum_creatinine", "risk_score"]].head()

## Encode Categorical Variables

In [ ]:
encoded_df = pd.get_dummies(
    df,
    columns=["sex", "smoking", "age_group"],
    dtype=int
)
encoded_df.head()

## Normalize Features

In [ ]:
scaled_df = encoded_df.copy()

scale_columns = [
    "age",
    "creatinine_phosphokinase",
    "ejection_fraction",
    "platelets",
    "serum_creatinine",
    "serum_sodium",
    "time",
    "risk_score"
]

scaler = MinMaxScaler()
scaled_df[scale_columns] = scaler.fit_transform(scaled_df[scale_columns])

scaled_df[scale_columns].head()

## Summary After Cleaning

In [ ]:
after_stats = df[summary_columns].agg(["mean", "median", "std"]).T
summary_comparison = pd.concat(
    {"Before": before_stats, "After": after_stats},
    axis=1
)
summary_comparison

## Validate Cleaning

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["age"], kde=True, ax=axes[0])
axes[0].set_title("Cleaned Age")
sns.boxplot(y=df["serum_creatinine"], ax=axes[1])
axes[1].set_title("Cleaned Serum Creatinine")
plt.tight_layout()
plt.show()

## Final Missing Value Check

In [ ]:
missing_after_cleaning = scaled_df.isna().sum()
print("Total missing values:", missing_after_cleaning.sum())
display(missing_after_cleaning.to_frame("Missing Values"))

## Save Cleaned Dataset

In [ ]:
scaled_df.to_csv("heart_failure_cleaned.csv", index=False)
print("Saved heart_failure_cleaned.csv")
scaled_df.head()

## Conclusion
The Heart Failure Clinical Records dataset was loaded, inspected, cleaned, enriched, encoded and normalized. Missing values were handled, outliers were removed, inconsistencies were checked, and the cleaned dataset was validated.